## PySpark Tutorial

```
Important:
  Use bash when working with conda environments

Some conda commands:

conda config --add channels conda-forge

# create environment with specific python version
conda create -n py37 python=3.7
conda activate py37
conda install jupyter ipython numpy pandas matplotlib

conda env list
conda activate py37
conda install pyspark
```

https://www.youtube.com/watch?v=MLXOy-OhWRY

Spark Programming Guide
- https://spark.apache.org/docs/latest/
- https://spark.apache.org/docs/latest/quick-start.html
- https://spark.apache.org/docs/latest/sql-programming-guide.html 
- https://spark.apache.org/docs/latest/rdd-programming-guide.html 
- https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html
- https://spark.apache.org/docs/latest/streaming-programming-guide.html 
- https://spark.apache.org/docs/latest/ml-guide.html 
- https://spark.apache.org/docs/latest/graphx-programming-guide.html
- https://spark.apache.org/docs/latest/api/python/getting_started/index.html


In [1]:
from pyspark import SparkContext, SparkConf

In [2]:
conf = SparkConf().setMaster('local')

In [3]:
sc = SparkContext(conf=conf)

In [4]:
# look at context, 
# click on link to the Spark UI, 
# click on tabs on top
sc

<SparkContext master=local appName=pyspark-shell>

In [7]:
rdd = sc.parallelize(range(10**6))

In [8]:
rdd.count()

1000000

In [12]:
sc.stop()
del sc

In [13]:
import pyspark
from pyspark import SparkContext
sc = SparkContext(master = 'local[2]')

In [27]:
# Inspect SparkContext
print( sc.version )              # SparkContext version 
print( sc.pythonVer )            # Python version 
print( sc.master )               # Master URL to connect to 
print( str(sc.sparkHome) )       # Path where Spark is installed on worker nodes 
print( str(sc.sparkUser()))      # name of the Spark User running SparkContext
print( sc.appName )              # application name 
print( sc.applicationId )        # application ID 
print( sc.defaultParallelism )   # default level of parallelism 
print( sc.defaultMinPartitions ) # minimum number of partitions for RDDs 

3.1.1
3.7
local
None
levselector
My app
local-1616978012283
1
1


In [17]:
sc.stop()
del sc

In [18]:
# Configuration
from pyspark import SparkConf, SparkContext
conf = (SparkConf()
         .setMaster("local")
         .setAppName("My app")
         .set("spark.executor.memory", "1g")) 

sc = SparkContext(conf = conf)

In [20]:
# Loading Data

# Parallelized Collections

rdd  = sc.parallelize([('a',7),('a',2),('b',2)])
rdd2 = sc.parallelize([('a',2),('d',1),('b',1)])
rdd3 = sc.parallelize(range(100))
rdd4 = sc.parallelize([("a",["x","y","z"]), ("b",["p", "r"])])

# External Data
# Read either one text file from HDFS, 
# a local file system 
# or any Hadoop-supported file system URI with textFile(), 
# or read in a directory of text files with wholeTextFiles().

textFile = sc.textFile("/tmp/lev/*.txt")
textFile2 = sc.wholeTextFiles("/tmp/lev/")

In [26]:
# Retrieving RDD Information
# Basic Information

print( rdd.getNumPartitions() ) # number of partitions 
  # 1
print( rdd.count() )            # Count RDD instances    
  # 3
print( rdd.countByKey() )       # Count RDD instances by key 
  # defaultdict(<class 'int'>, {'a': 2, 'b': 1})
    
print(rdd.countByValue())       # Count RDD instances by value 
  # defaultdict(<class 'int'>, {('a', 7): 1, ('a', 2): 1, ('b', 2): 1})

print( rdd.collectAsMap() )     # Return (key,value) pairs as a dictionary
  # {'a': 2,'b': 2}
    
print( rdd3.sum() )             # Sum of RDD elements 
  # 4950

print( sc.parallelize([]).isEmpty() ) # Check whether RDD is empty
  # True

1
3
defaultdict(<class 'int'>, {'a': 2, 'b': 1})
defaultdict(<class 'int'>, {('a', 7): 1, ('a', 2): 1, ('b', 2): 1})
{'a': 2, 'b': 2}
4950
True


In [28]:
# Summary

print( rdd3.max() )   # Maximum value of RDD elements
  # 99
print( rdd3.min() )   # Minimum value of RDD elements
  # 0
print( rdd3.mean() )  # Mean value of RDD elements
  # 49.5
print( rdd3.stdev() ) # Standard deviation of RDD elements
  # 28.866
print( rdd3.variance() )   # variance of RDD elements 
  # 833.25
print( rdd3.histogram(3) ) # histogram by bins  
  # ([0,33,66,99],[33,33,34])
print( rdd3.stats() )      # Summary statistics 
                           # (count, mean, stdev, max & min)

99
0
49.5
28.86607004772212
833.25
([0, 33, 66, 99], [33, 33, 34])
(count: 100, mean: 49.5, stdev: 28.86607004772212, max: 99.0, min: 0.0)


In [35]:
# Applying Functions

# Apply a function to each RDD element
print( rdd.map(lambda x: x+(x[1],x[0])).collect() )
  # [('a',7,7,'a'),('a',2,2,'a'),('b',2,2,'b')] 
    
# Apply a function to each RDD element and flatten the result
rdd5 = rdd.flatMap(lambda x: x+(x[1],x[0]))
print( rdd5.collect() )
  # ['a',7,7,'a','a',2,2,'a','b',2,2,'b'] 

# Apply a flatMap function to each (key,value) 
# pair of rdd4 without changing the keys
print( rdd4.flatMapValues(lambda x: x).collect() )
  # [('a','x'),('a','y'),('a','z'),('b','p'),('b','r')]

[('a', 7, 7, 'a'), ('a', 2, 2, 'a'), ('b', 2, 2, 'b')]
['a', 7, 7, 'a', 'a', 2, 2, 'a', 'b', 2, 2, 'b']
[('a', 'x'), ('a', 'y'), ('a', 'z'), ('b', 'p'), ('b', 'r')]


In [37]:
# Selecting Data
# -------------------------------
# Getting

# Return a list with all RDD elements 
print( rdd.collect() )
    # [('a', 7), ('a', 2), ('b', 2)] 

# Take first 2 RDD elements 
print( rdd.take(2) )
    # [('a', 7), ('a', 2)] 

# Take first RDD element 
print( rdd.first() )
    # ('a', 7) 

# Take top 2 RDD elements
print( rdd.top(2) )
    # [('b', 2), ('a', 7)]

[('a', 7), ('a', 2), ('b', 2)]
[('a', 7), ('a', 2)]
('a', 7)
[('b', 2), ('a', 7)]


In [38]:
# -------------------------------
# Sampling
# Return sampled subset of rdd3
print( rdd3.sample(False, 0.15, 81).collect() )
    # [3,4,27,31,40,41,42,43,60,76,79,80,86,97]

[3, 4, 27, 28, 35, 41, 43, 49, 53, 58, 85, 93]


In [48]:
# -------------------------------
# Filtering 
# Filter the RDD
print( rdd.filter(lambda x: "a" in x).collect() )
    # [('a',7),('a',2)] 
    
# Return distinct RDD values 
print( rdd5.distinct().collect() )
    # ['a',2,'b',7] 

# Return (key,value) RDD's keys
print( rdd.keys().collect() )
    # ['a', 'a', 'b']

[('a', 7), ('a', 2)]
['a', 7, 2, 'b']
['a', 'a', 'b']


In [53]:
# Iterating
# Apply a function to all RDD elements
def g(x): 
    print(x)
print( rdd.foreach(g) )
  # None
    
rdd.take(3)
  # ('a', 7) ('b', 2) ('a', 2)

None


[('a', 7), ('a', 2), ('b', 2)]

In [ ]:
# Reshaping Data

# -------------------------------
# Reducing
# Merge the rdd values for each key
print( rdd.reduceByKey(lambda x,y : x+y).collect() )
   # [('a',9),('b',2)] 

# Merge the rdd values
print( rdd.reduce(lambda a, b: a + b) )
   # ('a',7,'a',2,'b',2)

# -------------------------------
# Grouping by
print( rdd3.groupBy(lambda x: x % 2)
      .mapValues(list)
      .collect() )  # Return RDD of grouped values
print( rdd.groupByKey()
      .mapValues(list)
      .collect() )  # Group rdd by key
   # [('a',[7,2]),('b',[2])]

# -------------------------------
# Aggregating

seqOp = (lambda x,y: (x[0]+y,x[1]+1)) 
combOp = (lambda x,y:(x[0]+y[0],x[1]+y[1]))

# Aggregate RDD elements of each partition and then the results 
rdd3.aggregate((0,0),seqOp,combOp)
    # (4950,100) 

# Aggregate values of each RDD key
rdd.aggregateByKey((0,0),seqop,combop).collect() 
    # [('a',(9,2)), ('b',(2,1))] 

# Aggregate the elements of each partition, and then the results
rdd3.fold(0,add)
    # 4950 

# Merge the values for each key
rdd.foldByKey(0, add).collect() 
    # [('a',9),('b',2)] 

# Create tuples of RDD elements by applying a function
rdd3.keyBy(lambda x: x+x).collect()
    # 

In [33]:
# Mathematical Operations

# Return each rdd value not contained in rdd2
print( rdd.subtract(rdd2).collect() )
  # [('b',2),('a',7)] 
    
# Return each (key,value) pair of rdd2 with no matching key in rdd
print( rdd2.subtractByKey(rdd).collect() )
  # [('d', 1)] 

# Return the Cartesian product of rdd and rdd2
print( rdd.cartesian(rdd2).collect() )
  # [(('a', 7), ('a', 2)), 
  #  (('a', 7), ('d', 1)), 
  #  (('a', 7), ('b', 1)), 
  #  (('a', 2), ('a', 2)), 
  #  (('a', 2), ('d', 1)), 
  #  (('a', 2), ('b', 1)), 
  #  (('b', 2), ('a', 2)), 
  #  (('b', 2), ('d', 1)), 
  #  (('b', 2), ('b', 1))]

[('a', 7), ('b', 2)]
[('d', 1)]
[(('a', 7), ('a', 2)), (('a', 7), ('d', 1)), (('a', 7), ('b', 1)), (('a', 2), ('a', 2)), (('a', 2), ('d', 1)), (('a', 2), ('b', 1)), (('b', 2), ('a', 2)), (('b', 2), ('d', 1)), (('b', 2), ('b', 1))]


In [31]:
# Sort

# Sort RDD by given function
print( rdd2.sortBy(lambda x: x[1]).collect() )
  # [('d',1),('b',1),('a',2)] 

# Sort (key, value) RDD by key
print( rdd2.sortByKey().collect() )
  # [('a',2),('b',1),('d',1)]

[('d', 1), ('b', 1), ('a', 2)]
[('a', 2), ('b', 1), ('d', 1)]


In [ ]:
# Stopping SparkContext
sc.stop()